# Hands-on Lab 2 (Session 2): Transfer Learning Avanzado en NLP con Fine-Tuning Progresivo

**Nivel:** Intermedio / Avanzado  
**Dominio:** Procesamiento de Lenguaje Natural (NLP) - Clasificacion de Texto y Analisis de Sentimientos  
**Arquitectura:** Transformer Bidireccional (BERT) con KerasHub y Keras 3  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-02-transfer-learning/02-transfer-learning-nlp-advanced/02_transfer_learning_nlp_advanced.ipynb)

---

## 1. Fundamentos Teoricos: Transfer Learning Progresivo en Transformers

A diferencia del Feature Extraction basico en vision (donde simplemente se congelan todas las capas convolucionales y se entrena un cabezal), el Transfer Learning en modelos de lenguaje basados en **Transformers** presenta desafios especificos de estabilidad numerica y dinamica de gradientes.

### El Problema del Olvido Catastrofico (Catastrophic Forgetting)
Cuando se anade un nuevo cabezal de clasificacion con pesos aleatorios y se entrena inmediatamente todo el modelo con una tasa de aprendizaje estandar (ej. `1e-3`), los gradientes generados en las primeras iteraciones son extremadamente grandes y desordenados. Al propagarse hacia atras a traves de las capas de atencion, estos gradientes destruyen la rica representacion contextual y sintactica que el Transformer adquirio durante cientos de miles de horas de pre-entrenamiento.

### La Estrategia de Fine-Tuning Progresivo en 2 Fases
Para maximizar el rendimiento y evitar el olvido catastrofico, implementamos un protocolo riguroso en dos etapas:

```text
Fase 1: Warm-up del Cabezal (Feature Extraction)
┌────────────────────────────────────────────────────────┐
│ Transformer Backbone (CONGELADO - trainable=False)     │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ Nuevo Cabezal de Clasificacion (ENTRENABLE, lr=1e-3)   │
└────────────────────────────────────────────────────────┘
                           │
                           ▼
Fase 2: Fine-Tuning Progresivo (Descongelamiento Selectivo)
┌────────────────────────────────────────────────────────┐
│ Transformer Backbone (DESCONGELADO, lr=1e-5 o 100x menor)│
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ Cabezal de Clasificacion Adaptado (ENTRENABLE, lr=1e-5)│
└────────────────────────────────────────────────────────┘
```

### Objetivos Pedagogicos de este Laboratorio
- Preparar un dataset de clasificacion de texto en formato estructurado (entrenamiento y validacion).
- Cargar un backbone Transformer pre-entrenado (`bert_tiny_en_uncased`) mediante **KerasHub**.
- **Fase 1:** Congelar el backbone Transformer y entrenar unicamente el cabezal densificado a `learning_rate=1e-3`.
- **Fase 2:** Descongelar el Transformer y aplicar un ajuste fino conjunto a una tasa reducida (`learning_rate=2e-5`) con decaimiento de tasa de aprendizaje.
- Evaluar el impacto del descongelamiento comparando metricas antes y despues de la Fase 2.
- Calcular la **Matriz de Confusion** y el reporte detallado de **Precision, Recall y F1-Score** con scikit-learn.
- Probar el modelo con oraciones complejas que contienen negaciones, modismos y ambiguedades semanticas.


### Paso 1: Instalacion de Dependencias

Instalamos KerasHub, Keras 3 y bibliotecas de metricas y evaluacion cientifica:


In [ ]:
!pip install -q --upgrade keras keras-hub scikit-learn matplotlib numpy

### Paso 2: Configuracion del Backend y Carga de Modulos

Configuramos el backend numerico (JAX o TensorFlow) y cargamos las utilidades:


In [ ]:
import os
if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = "jax"  # o "tensorflow" o "torch"

import keras
import keras_hub
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(f"Keras version: {keras.__version__}")
print(f"KerasHub version: {keras_hub.__version__}")
print(f"Backend activo: {keras.backend.backend()}")

### Paso 3: Preparacion del Dataset de Resenas de Texto

Construimos un conjunto de datos representativo de analisis de sentimientos con textos que contienen opiniones positivas (clase 1) y negativas (clase 0), abarcando variabilidad lexica, expresiones coloquiales y negaciones gramaticales:


In [ ]:
# Dataset balanceado de entrenamiento
train_texts = [
    # Positivos (1)
    "This movie was an absolute masterpiece with brilliant acting and stunning visuals.",
    "I thoroughly enjoyed every minute of this film, truly exceptional directing.",
    "A captivating story that kept me on the edge of my seat from start to finish.",
    "Fantastic performance by the entire cast, easily one of the best films of the year.",
    "Incredible soundtrack and deeply moving character development, highly recommended.",
    "A heartwarming and brilliant cinematic journey with remarkable emotional depth.",
    "The cinematography is breathtaking and the storyline is wonderfully executed.",
    "An inspiring and delightful piece of art, I would gladly watch it again.",
    # Negativos (0)
    "A complete waste of time, terrible plot and painfully wooden acting throughout.",
    "Boring, predictable, and devoid of any genuine emotional connection.",
    "The dialogue was completely unnatural and the pacing dragged on for far too long.",
    "Disappointing from beginning to end with messy screenplay and shallow characters.",
    "I could barely sit through the first thirty minutes, utterly dreadful.",
    "Horrible directing and uninspired performances, do not waste your money.",
    "The special effects looked cheap and the storyline made absolutely no sense.",
    "An unmitigated disaster that fails on virtually every narrative level.",
]
train_labels = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

# Dataset de validacion con oraciones no vistas durante el entrenamiento
val_texts = [
    "Brilliant execution and thoroughly entertaining performances.",
    "A truly unforgettable film that exceeded all my expectations.",
    "Superb screenplay and magnificent visual direction.",
    "One of the finest cinematic achievements of recent times.",
    "Awful script, clumsy dialogue, and completely forgettable characters.",
    "A tedious and boring mess that fails to deliver any excitement.",
    "Completely unwatchable garbage with zero coherent plot.",
    "Frustratingly poor acting and thoroughly disjointed narrative.",
]
val_labels = [1, 1, 1, 1, 0, 0, 0, 0]

print(f"Muestras de entrenamiento: {len(train_texts)}")
print(f"Muestras de validacion: {len(val_texts)}")

### Paso 4: Carga del Modelo Transformer Pre-entrenado (BERT)

Utilizamos el preset `bert_tiny_en_uncased` de KerasHub:
- **Arquitectura:** 2 capas Transformer, 128 dimensiones de incrustacion oculta y 2 cabezales de auto-atencion.
- **Pesos Pre-entrenados:** Entrenado sobre BookCorpus y Wikipedia en ingles.
- **Tamano:** Menos de 18 MB, ideal para demostraciones agiles y ejecucion veloz en GPU o CPU.
- **Preprocessor Integrado:** Aplica automaticamente la tokenizacion subword (WordPiece) con tokens especiales `[CLS]` y `[SEP]`.


In [ ]:
classifier = keras_hub.models.BertClassifier.from_preset(
    "bert_tiny_en_uncased",
    num_classes=2,
    activation="softmax"
)
classifier.summary()

### Paso 5: Fase 1 - Congelamiento del Backbone (Feature Extraction Warm-up)

En esta primera fase, **congelamos el backbone** (`classifier.backbone.trainable = False`) para asegurar que solo los pesos recien inicializados de la capa de clasificacion superior se actualicen. Usamos una tasa de aprendizaje estandar de `1e-3`:


In [ ]:
# 1. Congelar el backbone Transformer
classifier.backbone.trainable = False

# 2. Compilar con tasa de aprendizaje estandar para el cabezal
classifier.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

print(f"Parametros entrenables (Fase 1): {np.sum([keras.ops.size(w) for w in classifier.trainable_weights]):,}")
print(f"Parametros congelados (Fase 1):   {np.sum([keras.ops.size(w) for w in classifier.non_trainable_weights]):,}")

### Paso 6: Entrenamiento de Fase 1 (Warm-up)

Entrenamos el nuevo cabezal durante 3 epocas para alinear los pesos de salida con las clases objetivo antes de tocar el Transformer:


In [ ]:
history_phase1 = classifier.fit(
    x=np.array(train_texts),
    y=np.array(train_labels),
    validation_data=(np.array(val_texts), np.array(val_labels)),
    epochs=3,
    batch_size=4
)

### Paso 7: Fase 2 - Descongelamiento Progresivo (Full Fine-Tuning con Tasa Reducida)

Una vez que el cabezal esta alineado, **descongelamos el backbone** (`classifier.backbone.trainable = True`).

> **Principio Critico de Estabilidad:** Reducimos la tasa de aprendizaje a `2e-5` (50 veces menor que en la Fase 1). Esta tasa minuscula permite realizar ajustes quirurgicos en las matrices de atencion del Transformer sin provocar olvido catastrofico:


In [ ]:
# 1. Descongelar todo el backbone Transformer
classifier.backbone.trainable = True

# 2. Recompilar con tasa de aprendizaje conservadora (discriminative fine-tuning)
classifier.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    metrics=["accuracy"]
)

print(f"Parametros entrenables (Fase 2): {np.sum([keras.ops.size(w) for w in classifier.trainable_weights]):,}")
print(f"Parametros congelados (Fase 2):   {np.sum([keras.ops.size(w) for w in classifier.non_trainable_weights]):,}")

### Paso 8: Entrenamiento de Fase 2 (Fine-Tuning de Extremo a Extremo)

Continuamos el entrenamiento durante 3 epocas adicionales refinando todas las capas del modelo:


In [ ]:
history_phase2 = classifier.fit(
    x=np.array(train_texts),
    y=np.array(train_labels),
    validation_data=(np.array(val_texts), np.array(val_labels)),
    epochs=3,
    batch_size=4
)

### Paso 9: Evaluacion Rigurosa con Matriz de Confusion y Reporte de Clasificacion

Calculamos predicciones sobre el conjunto de validacion y generamos la **Matriz de Confusion** y el reporte con Precision, Recall y F1-Score:


In [ ]:
val_preds_prob = classifier.predict(np.array(val_texts))
val_preds = np.argmax(val_preds_prob, axis=1)

# Reporte de clasificacion
target_names = ["Negativo (0)", "Positivo (1)"]
print("=== Reporte de Clasificacion ===\n")
print(classification_report(val_labels, val_preds, target_names=target_names))

# Visualizacion de Matriz de Confusion
cm = confusion_matrix(val_labels, val_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap="Blues")
plt.title("Matriz de Confusion - Fine-Tuning Progresivo")
plt.show()

### Paso 10: Inferencia Cualitativa sobre Frases Complejas (Sarcasmo y Negaciones)

Sometemos al modelo a oraciones linguisticamente dificiles que requieren entender el contexto global y las relaciones de atencion entre palabras:


In [ ]:
test_phrases = [
    "Not a single bad moment, absolutely loved it!",
    "I was expecting a masterpiece, but it was utterly boring.",
    "The acting was mediocre, but the stunning visual effects saved the entire experience.",
    "Never in my life have I seen such a poorly conceived catastrophe."
]

probs = classifier.predict(np.array(test_phrases))

print("=== Resultados de Inferencia Cualitativa ===\n")
for phrase, prob in zip(test_phrases, probs):
    sentiment = "Positivo" if prob[1] > prob[0] else "Negativo"
    conf = max(prob) * 100
    print(f"Texto:      \"{phrase}\"")
    print(f"Sentimiento: {sentiment} ({conf:.2f}% de confianza) [P(Pos)={prob[1]:.3f}, P(Neg)={prob[0]:.3f}]\n")

### Paso 11: Limpieza de Recursos de Memoria

Liberamos el modelo y los tensores de memoria para finalizar el laboratorio limpiamente:


In [ ]:
import gc
del classifier
gc.collect()
print("Recursos de memoria del Transformer liberados exitosamente.")